In [2]:
library(tidyverse)
library(repr)
library(paletteer)

In [14]:
tl_raw<- read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Tissue_Loss/T1-T4_Avg_Tissue_Loss.csv',show_col_types = FALSE)

Rows: 152 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (19): Date_InitialTag, Transect, NewTagNum, Species, immune_y/n, Date_Do...
dbl  (4): TransectNum, Size_Class, MaxDiameter, Height

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [15]:
tl_raw<-tl_raw %>%
    select(1:14) %>%
    select(-`immune_y/n`,-Height,-MaxDiameter,-Size_Class)
# create colony_id
tl_raw<-tl_raw %>%
    mutate(colony_id = paste0("T",TransectNum,"_",Species,'_',NewTagNum))

In [16]:
# combine with colony data for health statuses
colony<-read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Sample_Data/CBC_ColonyData.csv',show_col_types = FALSE)

Rows: 324 Columns: 59
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (51): Date_InitialTag, Transect, OldTagNum, NewTagNum, Species, Directio...
dbl  (6): TransectNum, Meter, Meters_90, Size_Class, MaxDiameter, Height
lgl  (2): 062019_Percentage, 102019_Percentage

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [17]:
colony<-colony %>%
    select(TransectNum,NewTagNum,Species,`062019_Condition`,`102019_Condition`,`052022_Condition`,`122022_Condition`) %>%
    filter(!(TransectNum %in% c('5','6')))
# create colony_id
colony<-colony %>%
    mutate(colony_id = paste0("T",TransectNum,"_",Species,'_',NewTagNum))

In [20]:
head(colony,2)

TransectNum,NewTagNum,Species,062019_Condition,102019_Condition,052022_Condition,122022_Condition,colony_id
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,SSID,Healthy,NA,Diseased,Diseased,T1_SSID_1
1,2,PAST,Healthy,NA,Healthy,Healthy,T1_PAST_2


In [22]:
# merge by colony id 
tl<-tl_raw %>% 
    left_join(colony %>% select(-TransectNum,-NewTagNum,-Species), by = "colony_id")

In [25]:
# select diseased colonies only
conditions <- c('Diseased','Dead')
tl_dis<-tl %>%
    filter(`052022_Condition` %in% conditions | `122022_Condition` %in% conditions)

In [40]:
# pivot
tl_dis_long <- tl_dis %>% 
  pivot_longer(
    cols = matches("^\\d{6}_(Condition|TL)$"),
    names_to = c("Date", ".value"),
    names_pattern = "^(\\d{6})_(Condition|TL)$"
  )

In [42]:
head(tl_dis_long,6)

Date_InitialTag,Transect,TransectNum,NewTagNum,Species,Date_DocumentedDisease,Date_DocumentedMortality,colony_id,Date,TL,Condition
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,062019,82.5,Healthy
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,052022,37.5,Diseased
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,122022,25,Diseased
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,102019,NA,NA
6/21/2019,CBC30N,1,3,SSID,5/21/2022,Diseased,T1_SSID_3,062019,95,Healthy
6/21/2019,CBC30N,1,3,SSID,5/21/2022,Diseased,T1_SSID_3,052022,95,Diseased


In [ ]:
# calculate rate of tl for diseased colonies
# change in TL over time
    # convert month years (Date) to date
    # calculate number of weeks in between time points
    # calculate change in TL between time points
    # May and Dec: change in TL / number of weeks

In [51]:
tl_dis_long<-tl_dis_long %>% 
  mutate(fulldate = as.character(my(Date)))